In [1]:
# Tìm thư mục gốc để notebook chạy được từ notebooks/ hoặc thư mục project.
from pathlib import Path
import sys

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src').is_dir()
)
sys.path.insert(0, str(project_root))
DATA_DIR = project_root / 'data'
MODEL_DIR = project_root / 'models'
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.metrics import classification_report

In [2]:
from src.models.fine_tuning import build_model, fine_tune_model

In [3]:
train_df = pd.read_csv(DATA_DIR / 'train_split.csv')
val_df = pd.read_csv(DATA_DIR / 'validation_split.csv')
test_df = pd.read_csv(DATA_DIR / 'test_split.csv')

# Ảnh được đổi về cùng kích thước và chuẩn hóa giống pipeline hiện tại.
def load_image(path, label):
    image = tf.io.read_file(path)
    image = tf.io.decode_jpeg(image, channels=3)
    image = tf.image.resize(image, (224, 224))
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def make_dataset(frame, training=False):
    paths = [str(DATA_DIR / name) for name in frame['file_name']]
    labels = frame['label'].to_numpy(dtype='float32')
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if training:
        dataset = dataset.shuffle(1000)
    dataset = dataset.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(16).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, training=True)
val_ds = make_dataset(val_df)
test_ds = make_dataset(test_df)
y_test = test_df['label'].to_numpy()

In [4]:
model = build_model()

# Pha 1: backbone đang đóng băng, chỉ học bộ phân loại mới.
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=2, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=1, min_lr=1e-6
)
head_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 50s 110ms/step - accuracy: 0.5230 - auc: 0.5325 - loss: 0.7003 - val_accuracy: 0.7227 - val_auc: 0.8357 - val_loss: 0.6714 - learning_rate: 1.0000e-04
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 47s 107ms/step - accuracy: 0.5641 - auc: 0.5922 - loss: 0.6791 - val_accuracy: 0.7533 - val_auc: 0.8648 - val_loss: 0.6565 - learning_rate: 1.0000e-04
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 54s 122ms/step - accuracy: 0.6023 - auc: 0.6566 - loss: 0.6595 - val_accuracy: 0.7813 - val_auc: 0.8729 - val_loss: 0.6431 - learning_rate: 1.0000e-04
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 187s 428ms/step - accuracy: 0.6379 - auc: 0.6997 - loss: 0.6456 - val_accuracy: 0.7973 - val_auc: 0.8780 - val_loss: 0.6308 - learning_rate: 1.0000e-04
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 79s 146ms/step - accuracy: 0.6619 - auc: 0.7220 - loss: 0.6374 - val_accuracy: 0.8007 - val_auc: 0.8797 - val_loss: 0.6196 - learning_rate: 1.0000e-04


In [5]:
# Pha 2: mở một phần MobileNet và tiếp tục học từ trọng số pha 1.
# Learning rate nhỏ giúp giữ lại kiến thức pretrained đã có.
model = fine_tune_model(model)
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_auc', mode='max', patience=2, restore_best_weights=True
)
reduce_lr = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.2, patience=1, min_lr=1e-7
)
fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=[early_stop, reduce_lr],
    shuffle=False,
)

Epoch 1/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 90s 203ms/step - accuracy: 0.6803 - auc: 0.7490 - loss: 0.6271 - val_accuracy: 0.8020 - val_auc: 0.8800 - val_loss: 0.6184 - learning_rate: 1.0000e-05
Epoch 2/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 173s 395ms/step - accuracy: 0.6787 - auc: 0.7507 - loss: 0.6265 - val_accuracy: 0.8007 - val_auc: 0.8800 - val_loss: 0.6172 - learning_rate: 1.0000e-05
Epoch 3/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 174s 330ms/step - accuracy: 0.6866 - auc: 0.7507 - loss: 0.6259 - val_accuracy: 0.8013 - val_auc: 0.8797 - val_loss: 0.6161 - learning_rate: 1.0000e-05
Epoch 4/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 45s 102ms/step - accuracy: 0.6754 - auc: 0.7443 - loss: 0.6270 - val_accuracy: 0.8020 - val_auc: 0.8804 - val_loss: 0.6150 - learning_rate: 1.0000e-05
Epoch 5/5
438/438 ━━━━━━━━━━━━━━━━━━━━ 54s 122ms/step - accuracy: 0.6864 - auc: 0.7506 - loss: 0.6249 - val_accuracy: 0.8027 - val_auc: 0.8799 - val_loss: 0.6139 - learning_rate: 1.0000e-05


In [ ]:
# Chỉ dùng tập test sau khi đã hoàn tất huấn luyện.
test_loss, test_accuracy, test_auc = model.evaluate(test_ds)
print(f'Test accuracy: {test_accuracy:.4f}; AUC: {test_auc:.4f}')

probabilities = model.predict(test_ds, verbose=1).ravel()
predictions = (probabilities >= 0.5).astype('int32')
print(classification_report(
    y_test, predictions, target_names=['Real', 'AI-generated'], digits=4
))

MODEL_DIR.mkdir(parents=True, exist_ok=True)
model.save(MODEL_DIR / 'fine_tuned_mobilenet.keras')

94/94 ━━━━━━━━━━━━━━━━━━━━ 10s 108ms/step - accuracy: 0.7953 - auc: 0.8738 - loss: 0.6188
Test accuracy: 0.7953; AUC: 0.8738
94/94 ━━━━━━━━━━━━━━━━━━━━ 20s 212ms/step
              precision    recall  f1-score   support

        Real     0.7786    0.8253    0.8013       750
AI-generated     0.8142    0.7653    0.7890       750

    accuracy                         0.7953      1500
   macro avg     0.7964    0.7953    0.7951      1500
weighted avg     0.7964    0.7953    0.7951      1500

